# FireST-GraphNet — End-to-End Training on Google Colab

**UCGIS 2026 Research Implementation**

This notebook provides a complete end-to-end pipeline for training
FireST-GraphNet on Google Colab (or any GPU environment).

## Architecture Overview

```
┌───────────── Encoder (28ch per timestep) ─────────────┐
│  fire(8 VIIRS) + aux(19 FirePred) + static(1 LULC)    │
│  → Multi-Source CNN + Dual-CBAM → GNN → LSTM           │
└────────────────────────────────────────────────────────┘
                        │
                        ▼
┌─── ForecastDecoder (semantically aligned 4ch) ────────┐
│  ALL sources → 4ch [temp, u, v, rh]                    │
│  → per-source adapter (4→32) → shared backbone → FiLM  │
│  → LSTM auto-regressive → SpatialHead                  │
│                                                         │
│  TRAINING:    FirePred dynamic 4ch extracted from 19ch  │
│  INFERENCE A: HRRR 4ch (TMP, UGRD, VGRD, RH)          │
│  INFERENCE B: WeatherNext2 4ch (⚠ placeholder)         │
└────────────────────────────────────────────────────────┘
```

## Semantic Alignment Design

The future-condition decoder branch uses **exactly 4 channels** for
**all** data sources, ensuring no semantic mismatch between training
and inference:

| Channel | Variable | FirePred Source | HRRR | WN2 |
|---------|----------|----------------|------|-----|
| 0 | Temperature (K) | band 5 | TMP | temperature_2m |
| 1 | U-wind (m/s) | speed(9)×sin(dir(10)) | UGRD | u10 |
| 2 | V-wind (m/s) | speed(9)×cos(dir(10)) | VGRD | v10 |
| 3 | Humidity (%) | band 7 | RH | rh_2m |

## Step 0 — Environment Setup

In [ ]:
# ── 0a. Check GPU availability ──
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu}  |  CUDA {torch.version.cuda}")
else:
    print("⚠ No GPU — training will be slow. Go to Runtime → Change runtime type → GPU.")

In [ ]:
# ── 0b. Install dependencies (Colab) ──
import subprocess, sys

# PyTorch Geometric
try:
    import torch_geometric
    print(f"PyG already installed: {torch_geometric.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'torch-geometric', '-q'])
    import torch_geometric
    print(f"Installed PyG: {torch_geometric.__version__}")

# Other deps
for pkg in ['pyyaml', 'tqdm', 'matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

In [ ]:
# ── 0c. Mount Google Drive (Colab only) ──
import os

IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(globals())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # WORKSPACE = where the firest_graphnet package lives (code)
    WORKSPACE = '/content/drive/MyDrive/FireST-GraphNet/FireST-GraphNet-colab'
    # DATA_ROOT = where raw data, dataset_train, dataset_val, outputs live
    DATA_ROOT = '/content/drive/MyDrive/FireST-GraphNet'
else:
    WORKSPACE = os.path.abspath('.')
    DATA_ROOT = os.path.abspath(os.path.join(WORKSPACE, '..'))

os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)
print(f"Workspace (code): {WORKSPACE}")
print(f"Data root:        {DATA_ROOT}")

In [ ]:
# ── 0d. Add package to path ──
import sys
if WORKSPACE not in sys.path:
    sys.path.insert(0, WORKSPACE)

# Quick import check
from firest_graphnet.models.firest_graphnet import FireSTGraphNet
from firest_graphnet.data.future_providers import (
    DYNAMIC_FUTURE_CHANNELS, FIREPRED_BAND_NAMES,
    extract_dynamic_from_firepred, extract_wind,
)
print(f"FireSTGraphNet loaded | Dynamic future channels: {DYNAMIC_FUTURE_CHANNELS}")
print(f"FirePred bands: {len(FIREPRED_BAND_NAMES)}")

## Step 1 — Preprocess Raw TS-SatFire Data

Raw data must be placed in Google Drive at:
```
MyDrive/FireST-GraphNet/ts-satfire_raw/
  <event_id>/          ← each wildfire event is one folder (e.g. 20562846)
    VIIRS_Day/         ← one .tif per day (8-band)
    VIIRS_Night/       ← one .tif per day (2-band)
    FirePred/          ← one .tif per day (19-band weather/topo/fuel)
    ESRI_LULC/         ← one static .tif (1-band land cover)
```

Running the cells below will generate:
```
dataset_train/
  af_train_img_seqtoseq_alll_1i_1.npy   (N, 8, 1, 256, 256)  VIIRS
  af_train_label_seqtoseq_alll_1i_1.npy (N, 3, 1, 256, 256)  BA/AF
  firepred_train.npy                     (N, 19, 1, 256, 256) weather+topo+fuel
  lulc_train.npy                         (N, 1, 256, 256)     land cover
dataset_val/   (same structure)
```

**Skip if you already have the npy files** — the cells auto-detect and skip.

In [ ]:
# ── 1a. Install rasterio (needed for reading GeoTIFF) ──
import subprocess, sys, os
from pathlib import Path

try:
    import rasterio
    print(f"rasterio already installed: {rasterio.__version__}")
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rasterio', '-q'])
    import rasterio
    print(f"Installed rasterio: {rasterio.__version__}")

# ── 1b. Locate raw data and existing npy files ──
data_root  = Path(DATA_ROOT)
raw_dir    = data_root / 'ts-satfire_raw'
train_dir  = data_root / 'dataset_train'
val_dir    = data_root / 'dataset_val'

# Auto-discover event IDs (subfolders that contain VIIRS_Day/)
if raw_dir.exists():
    event_ids = sorted([
        d.name for d in raw_dir.iterdir()
        if d.is_dir() and (d / 'VIIRS_Day').exists()
    ])
    print(f"\nRaw data: {raw_dir}")
    print(f"Found {len(event_ids)} event(s): {event_ids[:10]}{'...' if len(event_ids)>10 else ''}")
    for eid in event_ids[:5]:
        days = sorted((raw_dir / eid / 'VIIRS_Day').glob('*.tif'))
        has_fp = (raw_dir / eid / 'FirePred').exists()
        has_lulc = (raw_dir / eid / 'ESRI_LULC').exists()
        print(f"  {eid}: {len(days)} days | FirePred={'YES' if has_fp else 'NO'} | LULC={'YES' if has_lulc else 'NO'}")
    if len(event_ids) > 5:
        print(f"  ... and {len(event_ids)-5} more events")
else:
    event_ids = []
    print(f"WARNING: raw_dir not found: {raw_dir}")
    print("Please upload your event folders to Drive at ts-satfire_raw/<event_id>/")

# Check what npy files already exist
print("\nExisting npy files:")
for d in [train_dir, val_dir]:
    if d.exists():
        npys = sorted(d.glob('*.npy'))
        for f in npys:
            import numpy as np
            a = np.load(str(f), mmap_mode='r')
            print(f"  {d.name}/{f.name}: {a.shape}")
    else:
        print(f"  {d.name}/  — not found yet")

In [ ]:
# ── 1c. Run preprocessing (skip if npy already exists) ──
import numpy as np
from firest_graphnet.data.preprocess_raw import process_all_events, discover_event_ids

def _need_preprocess(out_dir: Path, mode: str) -> bool:
    """Return True if the img npy is missing or empty."""
    p = out_dir / f'af_{mode}_img_seqtoseq_alll_1i_1.npy'
    if not p.exists():
        return True
    a = np.load(str(p), mmap_mode='r')
    return a.shape[0] == 0

# ── Split events train/val (80/20). For a single event use all as train. ──
if len(event_ids) == 0:
    print("No event IDs found — skipping preprocessing.")
elif len(event_ids) == 1:
    # Only one event: use all windows for train, last 20% for val
    train_ids = event_ids
    val_ids   = event_ids   # overlap is fine for a demo
    print(f"Single event '{event_ids[0]}' — using for both train and val.")
else:
    split_idx  = max(1, int(len(event_ids) * 0.8))
    train_ids  = event_ids[:split_idx]
    val_ids    = event_ids[split_idx:]
    print(f"Train events ({len(train_ids)}): {train_ids}")
    print(f"Val   events ({len(val_ids)}):   {val_ids}")

PREPROCESS_KWARGS = dict(
    raw_dir     = raw_dir,
    ts_length   = 1,    # frames per window (matches existing npy format)
    interval    = 1,    # stride between windows
    image_size  = 256,
)

# ── Train ──────────────────────────────────────────────────────────────────
if len(event_ids) > 0:
    if _need_preprocess(train_dir, 'train'):
        print("\n=== Preprocessing TRAIN ===")
        process_all_events(
            out_dir   = train_dir,
            event_ids = train_ids,
            mode      = 'train',
            **PREPROCESS_KWARGS,
        )
    else:
        print("TRAIN npy already exists — skipping.")

    # ── Val ────────────────────────────────────────────────────────────────
    if _need_preprocess(val_dir, 'val'):
        print("\n=== Preprocessing VAL ===")
        process_all_events(
            out_dir   = val_dir,
            event_ids = val_ids,
            mode      = 'val',
            **PREPROCESS_KWARGS,
        )
    else:
        print("VAL npy already exists — skipping.")

In [ ]:
# ── 1d. Verify all output npy shapes + recommend T_in / horizon ──
import numpy as np
from pathlib import Path

print("Output npy files:")
N_train = 0
for d in [train_dir, val_dir]:
    if not d.exists():
        print(f"  {d.name}/  MISSING"); continue
    npys = sorted(d.glob('*.npy'))
    if not npys:
        print(f"  {d.name}/  EMPTY"); continue
    for f in npys:
        a = np.load(str(f), mmap_mode='r')
        tag = ''
        if 'img' in f.name:
            tag = '  <- VIIRS 8ch'
            if 'train' in f.name: N_train = a.shape[0]
        elif 'label' in f.name: tag = '  <- BA/AF_acc/AF 3ch'
        elif 'firepred' in f.name: tag = '  <- weather+topo+fuel 19ch  (REAL DATA)'
        elif 'lulc' in f.name:     tag = '  <- land cover 1ch  (REAL DATA)'
        print(f"  {d.name}/{f.name}: {a.shape}{tag}")

# ── Auto-recommend window settings ────────────────────────────────────────
if N_train > 0:
    print(f"\nTraining samples (frames): {N_train}")
    if N_train < 10:
        rec_Tin, rec_H = 1, 1
        print(f"  Very small dataset ({N_train} frames).")
        print(f"  Recommended: T_in={rec_Tin}, horizon={rec_H} => {N_train - rec_Tin - rec_H + 1} windows")
    elif N_train < 20:
        rec_Tin, rec_H = 3, 2
        print(f"  Small dataset ({N_train} frames).")
        print(f"  Recommended: T_in={rec_Tin}, horizon={rec_H} => {N_train - rec_Tin - rec_H + 1} windows")
    else:
        rec_Tin, rec_H = 10, 5
        print(f"  Standard dataset ({N_train} frames).")
        print(f"  Recommended: T_in={rec_Tin}, horizon={rec_H} => {N_train - rec_Tin - rec_H + 1} windows")
    # Export so Step 2a can use them
    RECOMMENDED_T_IN = rec_Tin
    RECOMMENDED_HORIZON = rec_H
    print(f"\nSet RECOMMENDED_T_IN={rec_Tin}, RECOMMENDED_HORIZON={rec_H} for Step 2a.")
else:
    RECOMMENDED_T_IN, RECOMMENDED_HORIZON = 3, 2
    print("No training npy found — using conservative defaults: T_in=3, horizon=2")

## Step 2 — Build Dataset & DataLoaders

The dataset emits:
- `af_hist` (T_in, 8, H, W) — VIIRS fire imagery
- `aux_hist` (T_in, 19, H, W) — FirePred auxiliary (full 19ch for encoder)
- `env_fut` (Horizon, **4**, H, W) — future dynamic env (**always 4ch**)
- `af_fut` (Horizon, 1, H, W) — ground truth
- `ba_init` (1, H, W) — burned area initial state
- `static` (1, H, W) — LULC

During training, `env_fut` is extracted from FirePred future frames:
- Channel 0: temperature (band 5)
- Channel 1: u-wind (derived from speed×sin(dir))
- Channel 2: v-wind (derived from speed×cos(dir))
- Channel 3: humidity (band 7)

In [ ]:
# ── 2a. Load config (auto-adjusted for dataset size) ──
import yaml, torch

cfg_path = Path(WORKSPACE) / 'firest_graphnet' / 'configs' / 'base.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Override for Colab training — data_root points to DATA_ROOT (not WORKSPACE)
cfg['data']['root'] = str(data_root)
cfg['forecast']['enabled'] = True
cfg['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Auto-adjust T_in / horizon based on dataset size ──────────────────────
# RECOMMENDED_T_IN / RECOMMENDED_HORIZON come from 1d cell above.
# For a full TS-SatFire dataset (192 events, 2000+ frames) the defaults
# T_in=10, horizon=5 are fine.  For a single event (8 frames) use smaller.
try:
    cfg['forecast']['T_in']     = RECOMMENDED_T_IN
    cfg['forecast']['horizon']  = RECOMMENDED_HORIZON
    print(f"Using dataset-sized window: T_in={RECOMMENDED_T_IN}, horizon={RECOMMENDED_HORIZON}")
except NameError:
    print(f"Using config defaults: T_in={cfg['forecast']['T_in']}, horizon={cfg['forecast']['horizon']}")

# ── Training hyper-params ──────────────────────────────────────────────────
cfg['training']['epochs']     = 50
cfg['training']['batch_size'] = 2  # 2 is safe for T4 GPU with T_in<=10
cfg['training']['lr']         = 1e-3

DEVICE = torch.device(cfg['device'])
fcfg   = cfg['forecast']
dcfg   = cfg['data']
print(f"Config loaded | Device: {DEVICE}")
print(f"Data root: {cfg['data']['root']}")
print(f"Forecast: T_in={fcfg['T_in']}, horizon={fcfg['horizon']}, future_ch={fcfg['future_channels']}")
print(f"Training: epochs={cfg['training']['epochs']}, batch={cfg['training']['batch_size']}, lr={cfg['training']['lr']}")

In [ ]:
# ── 2b. Create providers ──
from firest_graphnet.data.firepred_provider import build_firepred_provider

# FirePred provider for historical aux_hist (full 19ch) and future extraction
firepred_prov = build_firepred_provider(data_root=str(data_root))
print(f"FirePred provider: {type(firepred_prov).__name__} ({firepred_prov.channels}ch)")

# Static provider (LULC) — optional
try:
    from firest_graphnet.data.static_provider import build_static_provider
    static_prov = build_static_provider(data_root=str(data_root))
    print(f"Static provider: {type(static_prov).__name__}")
except Exception:
    static_prov = None
    print("Static provider: None (using zeros)")

In [ ]:
# ── 2c. Build datasets ──
from firest_graphnet.data.dataset_forecast import build_forecast_datasets
from torch.utils.data import DataLoader

fcfg = cfg['forecast']
dcfg = cfg['data']

train_ds, val_ds = build_forecast_datasets(
    data_root=str(data_root),
    T_in=fcfg['T_in'],
    horizon=fcfg['horizon'],
    label_channel=dcfg['label_channel'],
    val_split=dcfg['val_split'],
    seed=cfg['seed'],
    firepred_provider=firepred_prov,
    static_provider=static_prov,
)

BS = cfg['training']['batch_size']
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False,
                        num_workers=0, pin_memory=True)

print(f"Train: {len(train_ds)} samples, {len(train_loader)} batches")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")

In [ ]:
# ── 2d. Inspect a sample — verify semantic alignment ──
sample = train_ds[0]
print("Sample shapes:")
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:12s}: {tuple(v.shape)}")
    elif isinstance(v, str):
        print(f"  {k:12s}: '{v}'")
    elif isinstance(v, dict):
        print(f"  {k:12s}: {v}")

# Verify env_fut is 4ch
env_fut = sample['env_fut']
assert env_fut.shape[1] == 4, f"Expected 4ch env_fut, got {env_fut.shape[1]}ch"
print(f"\n✓ env_fut is {env_fut.shape[1]}ch canonical [temp, u, v, rh]")
print(f"  env_source: {sample['env_source']}")

## Step 3 — Model Construction

In [ ]:
# ── 3a. Build model ──
from firest_graphnet.models.firest_graphnet import FireSTGraphNet

FIRE_CH   = 8   # VIIRS Day(6) + Night(2)
AUX_CH    = 19  # FirePred (historical encoder)
STATIC_CH = 1   # LULC
FUT_CH    = cfg['forecast']['future_channels']  # 4 canonical

mcfg = cfg['model']
model = FireSTGraphNet(
    in_channels=FIRE_CH + AUX_CH + STATIC_CH,  # 28
    encoder_channels=mcfg['encoder_channels'],
    use_cbam=mcfg['use_cbam'],
    patch_size=mcfg['patch_size'],
    k_neighbors=mcfg['k_neighbors'],
    gnn_hidden=mcfg['gnn_hidden'],
    gnn_layers=mcfg['gnn_layers'],
    lstm_hidden=mcfg['lstm_hidden'],
    lstm_layers=mcfg['lstm_layers'],
    lstm_dropout=mcfg['lstm_dropout'],
    decoder_channels=mcfg['decoder_channels'],
    out_channels=mcfg['out_channels'],
    img_h=dcfg['img_size'],
    img_w=dcfg['img_size'],
    # Forecast mode
    forecast_mode=True,
    future_channels=FUT_CH,
    future_emb_dim=cfg['forecast']['future_emb_dim'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params:,} trainable parameters")
print(f"Encoder: {FIRE_CH}+{AUX_CH}+{STATIC_CH} = {FIRE_CH+AUX_CH+STATIC_CH}ch")
print(f"Future decoder: {FUT_CH}ch → per-source adapter → backbone → FiLM")

In [ ]:
# ── 3b. Quick forward pass sanity check ──
model.eval()
with torch.no_grad():
    B = 2
    dummy_af  = torch.randn(B, fcfg['T_in'], FIRE_CH, 64, 64, device=DEVICE)
    dummy_aux = torch.randn(B, fcfg['T_in'], AUX_CH, 64, 64, device=DEVICE)
    dummy_env = torch.randn(B, 5, FUT_CH, 64, 64, device=DEVICE)
    dummy_st  = torch.randn(B, STATIC_CH, 64, 64, device=DEVICE)

    # Test all three sources
    for src in ['firepred', 'hrrr', 'weathernext2']:
        out = model(dummy_af, aux_hist=dummy_aux, env_fut=dummy_env,
                    env_source=src, static=dummy_st)
        print(f"  [{src:14s}] → {tuple(out.shape)}")
        assert out.shape == (B, 5, 1, 64, 64)

print("✓ All source adapters working (4ch → 32 → backbone)")

## Step 4 — Training with PINN Constraints

The trainer uses `ForecastPINNLoss` which combines:
- **Supervised loss**: BCE per horizon step
- **PDE residual**: convection-diffusion-reaction with wind from `env_fut`
- **BA monotonicity**: burned area must never decrease
- **AF-BA consistency**: fire should not burn in already-burned area

Wind is extracted uniformly from `env_fut` channels 1,2 (u,v)
regardless of source — this works because all sources are
semantically aligned to [temp, u, v, rh].

In [ ]:
# ── 4a. Initialize trainer ──
from firest_graphnet.training.trainer import ForecastTrainer

trainer = ForecastTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=DEVICE,
    output_dir=str(data_root / 'outputs'),
)
print(f"Trainer ready | Epochs: {cfg['training']['epochs']}")
print(f"PINN: β_max={cfg['pinn']['beta_max']}, η_max={cfg['pinn']['eta_max']}, ξ_max={cfg['pinn']['xi_max']}")

In [ ]:
# ── 4b. Train! ──
trainer.fit()

## Step 5 — Inference with Different Weather Sources

Since all sources provide the same 4ch (temp, u, v, rh) to the decoder,
switching from FirePred (training) to HRRR (inference) is seamless —
just change the `env_source` parameter.

In [ ]:
# ── 5a. Load best model ──
ckpt_path = data_root / 'outputs' / 'best_model.pt'
if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best model (epoch {ckpt['epoch']}, IoU={ckpt['best_val_iou']:.4f})")
else:
    print("No checkpoint found — using current model weights")

In [ ]:
# ── 5b. Inference with HRRR future conditions ──
from firest_graphnet.data.future_providers import build_future_provider

# Try to load HRRR provider (falls back to placeholder if no data)
hrrr_prov = build_future_provider(
    mode='hrrr',
    data_root=str(data_root),
    spatial_size=(dcfg['img_size'], dcfg['img_size']),
)
print(f"HRRR provider: {type(hrrr_prov).__name__} ({hrrr_prov.channels}ch)")
print(f"Source name: '{hrrr_prov.source_name}'")

# Run inference on a val sample
model.eval()
sample = val_ds[0]

# Get HRRR future conditions (4ch)
fut_ids = tuple(range(fcfg['horizon']))
hrrr_fut = torch.from_numpy(
    hrrr_prov.get_future_env(fut_ids)
).float().unsqueeze(0).to(DEVICE)  # (1, H, 4, Hp, Wp)

af_hist  = sample['af_hist'].unsqueeze(0).to(DEVICE)
aux_hist = sample['aux_hist'].unsqueeze(0).to(DEVICE)
static   = sample['static'].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits_hrrr = model(
        af_hist, aux_hist=aux_hist, env_fut=hrrr_fut,
        env_source='hrrr', static=static,
    )
print(f"HRRR inference output: {tuple(logits_hrrr.shape)}")
print(f"Prediction range: [{logits_hrrr.min():.3f}, {logits_hrrr.max():.3f}]")

In [ ]:
# ── 5c. Compare predictions across sources ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Get predictions for all 3 sources
model.eval()
sample = val_ds[0]
af_h  = sample['af_hist'].unsqueeze(0).to(DEVICE)
aux_h = sample['aux_hist'].unsqueeze(0).to(DEVICE)
st    = sample['static'].unsqueeze(0).to(DEVICE)
env_f = sample['env_fut'].unsqueeze(0).to(DEVICE)  # 4ch from FirePred
gt    = sample['af_fut']  # (H, 1, Hp, Wp)

preds = {}
with torch.no_grad():
    for src in ['firepred', 'hrrr', 'weathernext2']:
        logits = model(af_h, aux_hist=aux_h, env_fut=env_f,
                       env_source=src, static=st)
        preds[src] = torch.sigmoid(logits).cpu().numpy()[0, 0, 0]  # first step

# Plot
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(gt[0, 0].numpy(), cmap='hot', vmin=0, vmax=1)
axes[0].set_title('Ground Truth (t+1)')
for i, (src, pred) in enumerate(preds.items()):
    axes[i+1].imshow(pred, cmap='hot', vmin=0, vmax=1)
    axes[i+1].set_title(f'{src}')
for ax in axes:
    ax.axis('off')
plt.suptitle('Semantic Alignment: same 4ch input → different adapter weights')
plt.tight_layout()
plt.savefig(str(data_root / 'outputs' / 'source_comparison.png'), dpi=150)
plt.show()
print("Saved source_comparison.png")

## Step 6 — Wind Extraction & Physics Verification

Since all sources have the same 4ch layout, wind extraction is trivial:
- Channel 1 = u-wind (m/s, eastward)
- Channel 2 = v-wind (m/s, northward)

No source-dependent branching needed.

In [ ]:
# ── 6a. Verify wind extraction ──
from firest_graphnet.data.future_providers import extract_wind

env_sample = sample['env_fut'].unsqueeze(0)  # (1, H, 4, Hp, Wp)

# Wind is channels 1,2 regardless of source
wind_fp = extract_wind(env_sample, 'firepred')
wind_hr = extract_wind(env_sample, 'hrrr')
wind_wn = extract_wind(env_sample, 'weathernext2')

assert torch.equal(wind_fp, wind_hr), "Wind should be same"
assert torch.equal(wind_fp, wind_wn), "Wind should be same"

print(f"Wind shape: {tuple(wind_fp.shape)}")
print(f"  u-wind range: [{wind_fp[0,:,0].min():.3f}, {wind_fp[0,:,0].max():.3f}]")
print(f"  v-wind range: [{wind_fp[0,:,1].min():.3f}, {wind_fp[0,:,1].max():.3f}]")
print("✓ Wind extraction: uniform across all sources")

In [ ]:
# ── 6b. PDE residual with wind ──
from firest_graphnet.physics.pde_residual import PDEResidual

pde = PDEResidual(D=0.01, mu=0.1)
A_curr = torch.sigmoid(logits_hrrr[0, 0:1])  # (1, 1, H, W)
A_next = torch.sigmoid(logits_hrrr[0, 1:2])  # (1, 1, H, W)
w_step = wind_fp[0, 0:1].cpu()                # (1, 2, H, W)

R_with_wind = pde(A_curr.cpu(), A_next.cpu(), wind=w_step)
R_no_wind   = pde(A_curr.cpu(), A_next.cpu())

print(f"PDE residual (with wind): ‖R‖² = {(R_with_wind**2).mean():.6f}")
print(f"PDE residual (no wind):   ‖R‖² = {(R_no_wind**2).mean():.6f}")
print("Wind adds convection term to the PDE, affecting the residual.")

## Step 7 — Dynamic Extraction Demo

Shows how 4ch is extracted from FirePred 19ch for training.

In [ ]:
# ── 7. FirePred 19ch → 4ch dynamic extraction ──
from firest_graphnet.data.future_providers import (
    extract_dynamic_from_firepred,
    FIREPRED_BAND_NAMES,
    FIREPRED_CLASSIFICATION,
    DYNAMIC_FUTURE_MAP,
)

print("FirePred 19-band classification:")
for category, bands in FIREPRED_CLASSIFICATION.items():
    names = [FIREPRED_BAND_NAMES[b] for b in bands]
    print(f"  {category:30s}: {names}")

print("\nCanonical 4ch future dynamic mapping:")
for ch, info in DYNAMIC_FUTURE_MAP.items():
    print(f"  Channel {ch}: {info['name']:<15s} ← {info}")

# Demo extraction
demo_19 = np.random.randn(19, 64, 64).astype(np.float32)
demo_4 = extract_dynamic_from_firepred(demo_19)
print(f"\nExtraction: {demo_19.shape} → {demo_4.shape}")
print(f"  ch0 (temp):   band5 value={demo_19[5,32,32]:.3f} → extracted={demo_4[0,32,32]:.3f}")
print(f"  ch3 (rh):     band7 value={demo_19[7,32,32]:.3f} → extracted={demo_4[3,32,32]:.3f}")

## Step 8 — Training Metrics & Visualization

In [ ]:
# ── 8a. Plot training curves ──
import csv

metrics_path = data_root / 'outputs' / 'metrics.csv'
if metrics_path.exists():
    with open(metrics_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)

    epochs = [int(r['epoch']) for r in rows]
    train_loss = [float(r.get('train_loss', 0)) for r in rows]
    val_loss = [float(r.get('val_loss', 0)) for r in rows]
    val_iou = [float(r.get('val_iou', 0)) for r in rows]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, train_loss, label='Train')
    ax1.plot(epochs, val_loss, label='Val')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.set_title('Training Loss')

    ax2.plot(epochs, val_iou, 'g-')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Val IoU')
    ax2.set_title('Validation IoU')

    plt.tight_layout()
    plt.savefig(str(data_root / 'outputs' / 'training_curves.png'), dpi=150)
    plt.show()
    print(f"Best val IoU: {max(val_iou):.4f} at epoch {epochs[val_iou.index(max(val_iou))]}")
else:
    print("No metrics.csv found — run training first.")

In [ ]:
# ── 8b. Per-horizon evaluation ──
from firest_graphnet.training.metrics import compute_metrics_per_horizon

model.eval()
all_logits, all_targets = [], []

with torch.no_grad():
    for batch in val_loader:
        af_h  = batch['af_hist'].to(DEVICE)
        aux_h = batch['aux_hist'].to(DEVICE)
        env_f = batch['env_fut'].to(DEVICE)
        st    = batch['static'].to(DEVICE)
        src   = batch.get('env_source', ['firepred'])[0]

        logits = model(af_h, aux_hist=aux_h, env_fut=env_f,
                       env_source=src, static=st)
        all_logits.append(logits.cpu())
        all_targets.append(batch['af_fut'])

logits_cat  = torch.cat(all_logits)
targets_cat = torch.cat(all_targets)

per_h = compute_metrics_per_horizon(logits_cat, targets_cat)
print(f"{'Step':>5} {'IoU':>8} {'F1':>8} {'Prec':>8} {'Recall':>8}")
print('-' * 45)
for i, m in enumerate(per_h):
    print(f"{i:>5} {m['iou']:>8.4f} {m['f1']:>8.4f} {m['precision']:>8.4f} {m['recall']:>8.4f}")

## Summary

This notebook demonstrated the complete FireST-GraphNet pipeline with
**semantic alignment**:

1. **Encoder**: 28ch per timestep (8 VIIRS + 19 FirePred + 1 LULC)
2. **Future decoder**: 4ch canonical [temp, u, v, rh] for ALL sources
3. **Training**: 4ch extracted from FirePred bands 5,9+10,7
4. **Inference**: 4ch direct from HRRR or WeatherNext2
5. **PINN**: wind from channels 1,2 (uniform extraction)
6. **No semantic mismatch** between training and inference

# 🔥 FireST-GraphNet — End-to-End Colab Notebook

**UCGIS 2026 — Spatio-Temporal Fire Spread Prediction with PINN Constraints**

## Architecture Overview

| Component | Training | Inference Mode A | Inference Mode B |
|-----------|----------|-----------------|------------------|
| **Encoder** | fire(8) + FirePred_hist(19) + LULC(1) = **28ch** | same | same |
| **Future env** | **FirePred 19ch** (“firepred” adapter) | **HRRR 4ch** (“hrrr” adapter) | **WeatherNext2 4ch** (“weathernext2” adapter) |
| **Decoder** | FutureConditionEncoder → FiLM → LSTM | same backbone, different adapter | same backbone, different adapter |
| **PINN** | wind from FirePred band 9/10 | wind from HRRR UGRD/VGRD | wind from WN2 u/v |

### Data Sources (TS-SatFire Paper)
- **VIIRS Day** (6 spectral): I1, I2, I3, M13, M15, M11
- **VIIRS Night** (2 thermal): I4 (3.74µm), I5 (11.45µm)
- **FirePred** (19 bands): DEM, slope, aspect, canopy, ERC, temp, dewpoint, RH, precip, wind speed, wind dir, fuel moisture, PDSI, fuel model, dead fuel, VPD, NDVI, soil temp, snow
- **ESRI LULC** (1 band): land-use / land-cover

### Key Design
**Training**: FirePred serves BOTH historical auxiliary AND future environmental conditions.  
**Inference**: HRRR (single-step) or WeatherNext2 (multi-step) REPLACE the future branch only.  
The decoder’s `FutureConditionEncoder` uses **source-specific 1×1 conv adapters** that project {19ch, 4ch, 4ch} to a shared 32ch feature space, followed by a shared backbone.

## 0 │ Setup & Dependencies

In [ ]:
# --- Colab Setup ---
import os, sys, subprocess, shutil

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch', 'torchvision', 'numpy', 'matplotlib',
                           'tqdm', 'pyyaml', 'requests'])
    from google.colab import drive
    drive.mount('/content/drive')

import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1 │ Locate / Clone Repository

In [ ]:
# Auto-locate project root
CANDIDATES = [
    '/content/drive/MyDrive/FireST-GraphNet',
    '/content/FireST-GraphNet',
    os.path.abspath(os.path.join(os.getcwd(), '..', '..')),  # local dev
]
PROJECT_ROOT = None
for c in CANDIDATES:
    if os.path.isdir(os.path.join(c, 'firest_graphnet')):
        PROJECT_ROOT = c
        break

if PROJECT_ROOT is None:
    if IN_COLAB:
        os.chdir('/content')
        if not os.path.isdir('FireST-GraphNet'):
            print('Cloning repo...')
            # !git clone <YOUR_REPO_URL> FireST-GraphNet
            raise FileNotFoundError(
                'Please upload or clone FireST-GraphNet to /content or Drive.'
            )
        PROJECT_ROOT = '/content/FireST-GraphNet'
    else:
        raise FileNotFoundError('Cannot locate FireST-GraphNet project root.')

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

## 1b │ Prepare Data

Check for pre-processed `.npy` files. If raw TS-SatFire TIFFs are present, run preprocessing.

In [ ]:
from pathlib import Path
import numpy as np

DATA_ROOT = Path(PROJECT_ROOT)
TRAIN_DIR = DATA_ROOT / 'dataset_train'
VAL_DIR   = DATA_ROOT / 'dataset_val'

def check_npy(d):
    """Check which processed .npy files exist."""
    if not d.exists():
        return {}
    return {
        'af_img':    sorted(d.glob('af_*_img_*.npy')),
        'af_label':  sorted(d.glob('af_*_label_*.npy')),
        'firepred':  sorted(d.glob('firepred_*.npy')),
        'lulc':      sorted(d.glob('lulc_*.npy')),
    }

for name, d in [('Train', TRAIN_DIR), ('Val', VAL_DIR)]:
    info = check_npy(d)
    print(f'\n{name} ({d}):')
    for k, v in info.items():
        status = f'{len(v)} file(s)' if v else 'MISSING'
        print(f'  {k:12s}: {status}')

# If raw TIFFs exist but npy missing, run preprocess
RAW_DIR = DATA_ROOT / 'raw_tssatfire'
if RAW_DIR.exists() and not check_npy(TRAIN_DIR).get('af_img'):
    print('\nRaw TIFFs found but npy missing — running preprocess_raw.py...')
    from firest_graphnet.data.preprocess_raw import preprocess_event_batch
    preprocess_event_batch(str(RAW_DIR), str(TRAIN_DIR))
    print('Preprocessing complete.')

## 1c │ (Optional) Download HRRR DOY Climatology

Only needed for **Inference Mode A** (HRRR replacement). Skip if you only need training.

In [ ]:
HRRR_CLIM = DATA_ROOT / 'hrrr_cache' / 'climatology'
if HRRR_CLIM.exists() and list(HRRR_CLIM.glob('doy_*.npy')):
    print(f'HRRR climatology found: {len(list(HRRR_CLIM.glob("doy_*.npy")))} DOY files')
else:
    print('HRRR climatology not found.  To download (optional):')
    print('  python download_hrrr.py')
    print('  This is only needed for HRRR inference mode (Mode A).')

## 2 │ Load & Inspect Dataset

In [ ]:
# Load processed npy arrays for inspection
info = check_npy(TRAIN_DIR)

if info.get('af_img'):
    imgs = np.load(str(info['af_img'][0]), mmap_mode='r')
    lbls = np.load(str(info['af_label'][0]), mmap_mode='r')
    print(f'AF images : {imgs.shape}  dtype={imgs.dtype}')
    print(f'AF labels : {lbls.shape}  dtype={lbls.dtype}')
else:
    print('No af_*_img npy found — using placeholder data for demo.')

if info.get('firepred'):
    fp = np.load(str(info['firepred'][0]), mmap_mode='r')
    print(f'FirePred  : {fp.shape}  dtype={fp.dtype}')
else:
    print('No firepred npy found — will use zeros.')

if info.get('lulc'):
    lc = np.load(str(info['lulc'][0]), mmap_mode='r')
    print(f'LULC      : {lc.shape}  dtype={lc.dtype}')
else:
    print('No LULC npy found — will use zeros.')

## 3 │ Visualise Raw Data

In [ ]:
import matplotlib.pyplot as plt

def show_sample(idx=0):
    info = check_npy(TRAIN_DIR)
    if not info.get('af_img'):
        print('No data to visualise.')
        return
    imgs = np.load(str(info['af_img'][0]), mmap_mode='r')
    lbls = np.load(str(info['af_label'][0]), mmap_mode='r')

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    # Band 0 (I1)
    axes[0].imshow(imgs[idx, 0, 0], cmap='hot')
    axes[0].set_title(f'VIIRS I1 (frame {idx})')
    # Band 6 (I4 night)
    axes[1].imshow(imgs[idx, 6, 0], cmap='inferno')
    axes[1].set_title('VIIRS I4 Night')
    # AF label
    axes[2].imshow(lbls[idx, 1, 0], cmap='Reds')
    axes[2].set_title('AF label')
    # BA label
    axes[3].imshow(lbls[idx, 0, 0], cmap='YlOrRd')
    axes[3].set_title('BA label')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_sample(0)

## 4 │ Configuration

In [ ]:
import yaml

cfg_path = Path(PROJECT_ROOT) / 'firest_graphnet' / 'configs' / 'base.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# ---- Override for forecast mode ----
cfg['forecast']['enabled'] = True
cfg['forecast']['T_in'] = 10
cfg['forecast']['horizon'] = 5           # shorter for demo
cfg['model']['in_channels'] = 28         # fire(8)+aux(19)+static(1)
cfg['model']['img_h'] = 256
cfg['model']['img_w'] = 256
cfg['training']['batch_size'] = 2
cfg['training']['epochs'] = 3            # quick demo
cfg['training']['mixed_precision'] = False

# Future condition channels (adapter pattern)
TRAIN_FUT_CH  = cfg['forecast'].get('train_future_channels', 19)
INFER_FUT_CH  = cfg['forecast'].get('inference_future_channels', 4)
FUTURE_EMB    = cfg['forecast'].get('future_emb_dim', 64)

print('Config:')
print(f'  Encoder in_channels    = {cfg["model"]["in_channels"]} (fire 8 + aux 19 + static 1)')
print(f'  train_future_channels  = {TRAIN_FUT_CH} (FirePred 19ch)')
print(f'  infer_future_channels  = {INFER_FUT_CH} (HRRR / WN2 4ch)')
print(f'  future_emb_dim         = {FUTURE_EMB}')
print(f'  T_in={cfg["forecast"]["T_in"]}  horizon={cfg["forecast"]["horizon"]}')
print(f'  batch_size={cfg["training"]["batch_size"]}  epochs={cfg["training"]["epochs"]}')

## 5a │ Build Historical Providers

- **FirePredProvider** (19ch): historical auxiliary data (weather/topo/fuel)
- **LULCStaticProvider** (1ch): land-use / land-cover

In [ ]:
from firest_graphnet.data.firepred_provider import build_firepred_provider
from firest_graphnet.data.static_provider import build_static_provider

img_h = cfg['model'].get('img_h', 256)
img_w = cfg['model'].get('img_w', 256)
spatial = (img_h, img_w)

firepred_prov = build_firepred_provider(
    data_root=str(DATA_ROOT), channels=19, spatial_size=spatial,
)
static_prov = build_static_provider(
    data_root=str(DATA_ROOT), channels=1, spatial_size=spatial,
)
print(f'FirePred provider: {type(firepred_prov).__name__} ({firepred_prov.channels}ch)')
print(f'Static provider:   {type(static_prov).__name__}')

## 5b │ Build Future Condition Providers

| Provider | Source | Channels | Usage |
|----------|--------|----------|-------|
| `TSSatFireFutureProvider` | FirePred npy | 19 | **Training** |
| `HRRRFutureProvider` | DOY climatology | 4 | Inference Mode A |
| `WeatherNext2FutureProvider` | Google API | 4 | Inference Mode B |

During training, the dataset uses FirePred for BOTH `aux_hist` and `env_fut`.  
No external future provider is needed — the dataset reads future frames directly from the same FirePred npy.

In [ ]:
from firest_graphnet.data.future_providers import (
    build_future_provider,
    PlaceholderFutureProvider,
    extract_wind,
)

# For training: future_provider=None means dataset uses firepred_provider for env_fut
# This is the correct UCGIS-2026 design: train on TS-SatFire observed conditions.
train_future_provider = None  # ← dataset will use firepred_provider.get_window(fut_ids)

# For inference evaluation (built later in Section 7):
# hrrr_future_provider = build_future_provider('hrrr', data_root=str(DATA_ROOT), spatial_size=spatial)
# wn2_future_provider  = build_future_provider('weathernext2', spatial_size=spatial)

print('Training future env: FirePred 19ch (from firepred_provider, same npy)')
print('Inference future env: HRRR 4ch / WeatherNext2 4ch (built in Section 7)')

## 5c │ Build Datasets & DataLoaders

In [ ]:
from firest_graphnet.data.dataset_forecast import build_forecast_datasets
from torch.utils.data import DataLoader

fcfg = cfg['forecast']
dcfg = cfg['data']

train_ds, val_ds = build_forecast_datasets(
    data_root=str(DATA_ROOT),
    T_in=fcfg['T_in'],
    horizon=fcfg['horizon'],
    label_channel=dcfg.get('label_channel', 1),
    val_split=dcfg.get('val_split', 0.1),
    firepred_provider=firepred_prov,
    static_provider=static_prov,
    future_provider=train_future_provider,  # None = FirePred future
)

BS = cfg['training']['batch_size']
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=0)

print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples, {len(val_loader)} batches')

# Inspect one sample
sample = train_ds[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:12s}: {tuple(v.shape)}  dtype={v.dtype}')
    else:
        print(f'  {k:12s}: {v}')

## 5d │ Build Model

The decoder’s `FutureConditionEncoder` has three adapters:
- `"firepred"`: Conv2d(19→32) — used during **training**
- `"hrrr"`: Conv2d(4→32) — used during **inference Mode A**
- `"weathernext2"`: Conv2d(4→32) — used during **inference Mode B**

All adapters feed into a **shared backbone** (32→64→pool→emb).

In [ ]:
from firest_graphnet.models.firest_graphnet import FireSTGraphNet

mcfg = cfg['model']

model = FireSTGraphNet(
    in_channels=mcfg['in_channels'],             # 28 = fire(8) + aux(19) + static(1)
    encoder_channels=mcfg['encoder_channels'],
    use_cbam=mcfg.get('use_cbam', True),
    patch_size=mcfg['patch_size'],
    k_neighbors=mcfg['k_neighbors'],
    gnn_hidden=mcfg['gnn_hidden'],
    gnn_layers=mcfg['gnn_layers'],
    lstm_hidden=mcfg['lstm_hidden'],
    lstm_layers=mcfg['lstm_layers'],
    lstm_dropout=mcfg.get('lstm_dropout', 0.1),
    decoder_channels=mcfg['decoder_channels'],
    out_channels=mcfg.get('out_channels', 1),
    img_h=mcfg.get('img_h', 256),
    img_w=mcfg.get('img_w', 256),
    # Forecast mode with adapter pattern
    forecast_mode=True,
    train_future_channels=TRAIN_FUT_CH,          # 19 (FirePred)
    inference_future_channels=INFER_FUT_CH,       # 4  (HRRR / WN2)
    future_emb_dim=FUTURE_EMB,                    # 64
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {n_params:,} params ({n_train:,} trainable)')
print(f'Forecast mode: {model.forecast_mode}')
print(f'Decoder FutureConditionEncoder adapters: {list(model.decoder.future_enc.adapters.keys())}')

## 6 │ Train (FirePred 19ch Future Conditioning)

The model trains on **TS-SatFire observed multimodal sequences**:  
- **Encoder**: fire(8) + FirePred_hist(19) + LULC(1) = 28ch per timestep  
- **Decoder**: FirePred_fut(19) → `"firepred"` adapter → shared backbone → FiLM  
- **PINN**: wind from FirePred band 9 (speed) + band 10 (direction)

In [ ]:
from firest_graphnet.training.trainer import ForecastTrainer

trainer = ForecastTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
    output_dir=str(DATA_ROOT / 'outputs'),
)

trainer.fit()

## 6b │ (Optional) Load Saved Checkpoint

In [ ]:
CKPT = DATA_ROOT / 'outputs' / 'best_model.pt'
if CKPT.exists():
    ckpt = torch.load(str(CKPT), map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded checkpoint from epoch {ckpt.get("epoch", "?")} '
          f'(val IoU={ckpt.get("best_val_iou", "?"):.4f})')
else:
    print('No checkpoint found — using current model weights.')

## 7a │ Evaluate — FirePred (Training Conditions)

Uses the **same TS-SatFire environmental data** that was seen during training.  
This is the baseline — the model should perform best here.

In [ ]:
from firest_graphnet.training.metrics import compute_metrics, metrics_summary_str

def evaluate_with_source(model, loader, env_source, device, future_provider=None):
    """Run evaluation with a specific env_source adapter.
    
    If future_provider is given, it replaces env_fut from the dataloader.
    """
    model.eval()
    totals = {'iou': 0, 'f1': 0, 'precision': 0, 'recall': 0}
    n = 0
    with torch.no_grad():
        for batch in loader:
            af_hist  = batch['af_hist'].to(device)
            aux_hist = batch['aux_hist'].to(device)
            static   = batch['static'].to(device)
            af_fut   = batch['af_fut'].to(device)
            
            if future_provider is not None:
                # Override env_fut with external provider data
                B, H_steps = af_hist.shape[0], af_fut.shape[1]
                Hp, Wp = af_hist.shape[3], af_hist.shape[4]
                env_np = future_provider.get_future_env(
                    tuple(range(H_steps)), horizon=H_steps
                )  # (H, C, Hp, Wp)
                env_t = torch.from_numpy(env_np.copy()).float().to(device)
                # Broadcast to batch
                env_fut = env_t.unsqueeze(0).expand(B, -1, -1, -1, -1)
            else:
                env_fut = batch['env_fut'].to(device)
            
            logits = model(
                af_hist,
                aux_hist=aux_hist,
                env_fut=env_fut,
                env_source=env_source,
                static=static,
            )
            m = compute_metrics(logits, af_fut)
            for k in totals:
                totals[k] += m[k]
            n += 1
    
    avg = {k: v / max(n, 1) for k, v in totals.items()}
    return avg

# Evaluate with FirePred (training conditions)
fp_metrics = evaluate_with_source(model, val_loader, 'firepred', device)
print(f'\n=== Eval: FirePred 19ch (training) ===')
for k, v in fp_metrics.items():
    print(f'  {k}: {v:.4f}')

## 7b │ Evaluate — HRRR Single-Step Inference (Mode A)

Replace future env with HRRR 4ch (TMP, UGRD, VGRD, RH).  
The decoder switches to the `"hrrr"` adapter (Conv2d 4→32).

In [ ]:
# Build HRRR future provider (falls back to placeholder if climatology not downloaded)
hrrr_prov = build_future_provider('hrrr', data_root=str(DATA_ROOT), spatial_size=spatial)
print(f'HRRR provider: {type(hrrr_prov).__name__} ({hrrr_prov.channels}ch)')

hrrr_metrics = evaluate_with_source(
    model, val_loader, 'hrrr', device, future_provider=hrrr_prov,
)
print(f'\n=== Eval: HRRR 4ch (inference Mode A) ===')
for k, v in hrrr_metrics.items():
    print(f'  {k}: {v:.4f}')

## 7c │ Evaluate — WeatherNext2 Multi-Step Inference (Mode B)

Replace future env with WeatherNext2 4ch (TMP, u10, v10, RH).  
The decoder switches to the `"weathernext2"` adapter.

In [ ]:
wn2_prov = build_future_provider('weathernext2', spatial_size=spatial)
print(f'WN2 provider: {type(wn2_prov).__name__} ({wn2_prov.channels}ch)')

wn2_metrics = evaluate_with_source(
    model, val_loader, 'weathernext2', device, future_provider=wn2_prov,
)
print(f'\n=== Eval: WeatherNext2 4ch (inference Mode B) ===')
for k, v in wn2_metrics.items():
    print(f'  {k}: {v:.4f}')

# Summary comparison
print(f'\n{"="*50}')
print(f'{"Source":<20} {"IoU":>8} {"F1":>8} {"Prec":>8} {"Rec":>8}')
print(f'{"-"*50}')
for name, m in [('FirePred (train)', fp_metrics),
                ('HRRR (Mode A)', hrrr_metrics),
                ('WN2 (Mode B)', wn2_metrics)]:
    print(f'{name:<20} {m["iou"]:8.4f} {m["f1"]:8.4f} {m["precision"]:8.4f} {m["recall"]:8.4f}')
print(f'{"="*50}')

## 8 │ Visualise Predictions

In [ ]:
def visualize_predictions(model, loader, env_source, device, title='', n_samples=2):
    """Show prediction vs ground truth for a few samples."""
    model.eval()
    batch = next(iter(loader))
    af_hist  = batch['af_hist'][:n_samples].to(device)
    aux_hist = batch['aux_hist'][:n_samples].to(device)
    env_fut  = batch['env_fut'][:n_samples].to(device)
    af_fut   = batch['af_fut'][:n_samples].to(device)
    static   = batch['static'][:n_samples].to(device)

    with torch.no_grad():
        logits = model(
            af_hist, aux_hist=aux_hist, env_fut=env_fut,
            env_source=env_source, static=static,
        )
    probs = torch.sigmoid(logits).cpu().numpy()
    gt = af_fut.cpu().numpy()

    n_steps = min(5, probs.shape[1])
    fig, axes = plt.subplots(n_samples * 2, n_steps, figsize=(3 * n_steps, 3 * n_samples * 2))
    if n_samples == 1:
        axes = axes[np.newaxis, :] if axes.ndim == 1 else axes

    for i in range(n_samples):
        for t in range(n_steps):
            axes[2*i, t].imshow(probs[i, t, 0], cmap='hot', vmin=0, vmax=1)
            axes[2*i, t].set_title(f'Pred h={t}' if i == 0 else '')
            axes[2*i, t].axis('off')
            axes[2*i+1, t].imshow(gt[i, t, 0], cmap='hot', vmin=0, vmax=1)
            axes[2*i+1, t].set_title(f'GT h={t}' if i == 0 else '')
            axes[2*i+1, t].axis('off')

    fig.suptitle(f'{title} [{env_source}]', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_predictions(model, val_loader, 'firepred', device, title='FirePred (Training)')

## 9 │ Physics Constraint Analysis

In [ ]:
from firest_graphnet.physics.constraints import PhysicsConstraints
from firest_graphnet.physics.ba_update import BAStateUpdate

phys = PhysicsConstraints()
ba_update = BAStateUpdate(gamma=cfg['pinn'].get('gamma', 0.3))

# Get a batch
batch = next(iter(val_loader))
af_hist  = batch['af_hist'].to(device)
aux_hist = batch['aux_hist'].to(device)
env_fut  = batch['env_fut'].to(device)
af_fut   = batch['af_fut'].to(device)
ba_init  = batch['ba_init'].to(device)
static   = batch['static'].to(device)

model.eval()
with torch.no_grad():
    logits = model(
        af_hist, aux_hist=aux_hist, env_fut=env_fut,
        env_source='firepred', static=static,
    )

# Physics losses
phys_losses = phys(logits, ba_init)
print('Physics constraint losses:')
for k, v in phys_losses.items():
    print(f'  {k}: {v:.6f}')

# BA update with monotonicity check
ba_seq = ba_update(logits, ba_init)
for h in range(ba_seq.shape[1] - 1):
    diff = (ba_seq[:, h+1] - ba_seq[:, h]).min().item()
    status = '\u2713' if diff >= -1e-6 else '\u2717'
    print(f'  BA mono h={h}\u2192{h+1}: min_diff={diff:.6f} {status}')

# Wind extraction from FirePred env_fut
wind = extract_wind(env_fut, 'firepred')
if wind is not None:
    print(f'\nWind field from FirePred: {tuple(wind.shape)}')
    print(f'  u range: [{wind[:,:,0].min():.3f}, {wind[:,:,0].max():.3f}] m/s')
    print(f'  v range: [{wind[:,:,1].min():.3f}, {wind[:,:,1].max():.3f}] m/s')

## 10 │ Export & Save

In [ ]:
# Save final model
OUT_DIR = DATA_ROOT / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

save_path = OUT_DIR / 'final_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': cfg,
    'architecture': {
        'encoder_in_channels': cfg['model']['in_channels'],
        'train_future_channels': TRAIN_FUT_CH,
        'inference_future_channels': INFER_FUT_CH,
        'future_emb_dim': FUTURE_EMB,
        'adapters': ['firepred', 'hrrr', 'weathernext2'],
    },
}, str(save_path))
print(f'Model saved to: {save_path}')

# Copy to Drive if on Colab
if IN_COLAB:
    import shutil
    drive_out = Path('/content/drive/MyDrive/FireST-GraphNet/outputs')
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(save_path), str(drive_out / save_path.name))
    print(f'Copied to Drive: {drive_out / save_path.name}')

print('\n\u2705 Done!')

# FireST-GraphNet: Wildfire Progression Prediction via a Temporal Graph Neural Network with Sequence Learning

Architecture: CNN+Dual-CBAM → GNN → LSTM → Forecast-Conditioned Decoder (FiLM)  
Supports single-step (H=1) and multi-horizon (H=20) prediction with PINN physics constraints.

**Multi-source data pipeline (per TS-SatFire paper + UCGIS2026 abstract):**
| Stream | Source | Channels | Role |
|---|---|---|---|
| Fire imagery (historical) | `af_*_img_*.npy` — VIIRS Day 6 spectral (I1,I2,I3,M13,M15,M11) + Night 2 thermal (I4,I5) | 8 | CNN+GNN+LSTM backbone — fire observation |
| Auxiliary (historical) | `firepred_*.npy` — FirePred weather/topo/fuel (DEM, slope, aspect, canopy, ERC, temp, dewpoint, RH, precip, wind speed/dir, fuel moisture, PDSI, fuel model, dead fuel, VPD, NDVI, soil temp, snow) | 19 | Encoder input — meteorological + terrain |
| Static environment | `lulc_*.npy` — ESRI Land-Use / Land-Cover | 1 | Encoder input — vegetation / land cover |
| **Encoder total per timestep** | concat(fire + aux + static) | **28** | Multi-source fusion via Dual-CBAM |
| Future weather (decoder) | HRRR DOY climatology (TMP, UGRD, VGRD, RH) | 4 | FiLM decoder conditioning |
| Future weather (production) | Google WeatherNext2 | 4+ | FiLM decoder conditioning |
| Ground-truth targets | `af_*_label_*.npy` ch 1 | 1 (binary AF mask) | Loss supervision |

---

## Usage

1. Upload `FireST-GraphNet-colab/` folder to Google Drive under `MyDrive/`
2. Upload processed data to `MyDrive/FireST-GraphNet/dataset_train/` and `dataset_val/`  
   *(or keep raw TS-SatFire at `ts-satfire_raw/` and Step 1b will preprocess automatically)*
3. Open this notebook in Google Colab and run all cells sequentially

> **Step 1** auto-detects code & data paths.  
> **Step 1b** checks for processed data, downloads HRRR climatology (~1 MB), and preprocesses raw TS-SatFire TIFFs (VIIRS + FirePred + LULC) if available.  
> **Step 4** sets `weather_provider = "hrrr"` — used for FiLM decoder future weather conditioning.  
> FirePred 19ch auxiliary data is loaded by `firepred_provider` (falls back to zeros if `firepred_*.npy` not available).

## 0. Setup & Install Dependencies

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pyyaml tqdm matplotlib
    print("Colab environment ✓")
else:
    print("Local environment ✓")

import torch

print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

## Step 1 — Mount Drive & Auto-Locate Paths

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Step 1: Mount Drive + auto-locate CODE_ROOT / DATA_ROOT
# ════════════════════════════════════════════════════════════════════════
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# ── 1) Locate firest_graphnet package (CODE_ROOT) ─────────────────────
code_candidates = [
    Path("/content/drive/MyDrive/FireST-GraphNet-colab"),
    Path("/content/drive/MyDrive/FireST-GraphNet"),
    Path("/content"),
    Path(r"C:\Users\22125\Desktop\FireST-GraphNet"),       # local Windows
    Path.home() / "FireST-GraphNet",                        # local Mac/Linux
]

CODE_ROOT = None
for c in code_candidates:
    if (c / "firest_graphnet" / "__init__.py").exists():
        CODE_ROOT = c
        break

# Fallback: recursive search on Drive
if CODE_ROOT is None and IN_COLAB:
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        hits = list(drive_root.rglob("firest_graphnet/__init__.py"))
        if hits:
            CODE_ROOT = hits[0].parent.parent

if CODE_ROOT is None:
    raise RuntimeError(
        "Cannot find firest_graphnet package!\n"
        "Please upload FireST-GraphNet-colab/ to Google Drive."
    )

# ── 2) Locate data directory (DATA_ROOT) ──────────────────────────────
def has_data_markers(root: Path) -> bool:
    return any([
        (root / "dataset_train").exists(),
        (root / "dataset_val").exists(),
        (root / "ts-satfire_raw").exists(),
        (root / "hrrr_cache").exists(),
    ])

if IN_COLAB:
    data_candidates = [
        Path("/content/drive/MyDrive/FireST-GraphNet"),
        CODE_ROOT,
        Path("/content/drive/MyDrive/FireST-GraphNet-colab"),
        Path("/content"),
    ]
else:
    data_candidates = [CODE_ROOT]

DATA_ROOT = None
for d in data_candidates:
    if d.exists() and has_data_markers(d):
        DATA_ROOT = d
        break

if DATA_ROOT is None:
    if IN_COLAB:
        DATA_ROOT = Path("/content/drive/MyDrive/FireST-GraphNet")
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
    else:
        DATA_ROOT = CODE_ROOT

# ── 3) Add package to Python path ─────────────────────────────────────
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
os.chdir(str(CODE_ROOT))

import firest_graphnet

print(f"Code root : {CODE_ROOT}")
print(f"Data root : {DATA_ROOT}")
print("firest_graphnet loaded ✓")

## Step 1b — Prepare Data (TS-SatFire + FirePred + LULC + HRRR)

**Logic:**
1. Check for processed fire data (`dataset_train/` and `dataset_val/` with `af_*.npy`)
2. If missing → preprocess from raw TS-SatFire GeoTIFFs at `ts-satfire_raw/`
3. Build HRRR DOY climatology for FiLM decoder future weather (29 dates, ~1 MB)
4. Generate FirePred 19ch auxiliary and ESRI LULC static features from raw data if available

| Dataset | Source | Channels | Role |
|---|---|---|---|
| VIIRS imagery | `ts-satfire_raw/<event>/VIIRS_Day/*.tif` + `VIIRS_Night/*.tif` | 8 (6 spectral + 2 thermal) | Fire observation (encoder backbone) |
| FirePred auxiliary | `ts-satfire_raw/<event>/FirePred/*.tif` | 19 (weather/topo/fuel) | Historical context (encoder input) |
| ESRI LULC | `ts-satfire_raw/<event>/ESRI_LULC/*.tif` | 1 (land cover class) | Static features (encoder input) |
| HRRR Climatology | NOAA archive (`herbie`) | 4 (TMP/UGRD/VGRD/RH) | Future weather (FiLM decoder) |
| WeatherNext2 | Google (restricted) | 4+ | Future weather (auto-fallback to zeros) |

In [ ]:
"""
Step 1b — Prepare TS-SatFire processed data + HRRR climatology.

Raw TS-SatFire is assumed to be at DATA_ROOT / ts-satfire_raw/
(same structure as the Kaggle dataset: <event_id>/VIIRS_Day/*.tif, etc.)
"""
import subprocess, shutil, importlib
from pathlib import Path

# ── Config ───────────────────────────────────────────────────────────────
FORCE_REPROCESS = False   # True → force re-preprocessing even if npy exists

train_dir = Path(DATA_ROOT) / "dataset_train"
val_dir   = Path(DATA_ROOT) / "dataset_val"
raw_dir   = Path(DATA_ROOT) / "ts-satfire_raw"
clim_dir  = Path(DATA_ROOT) / "hrrr_cache" / "climatology"

required_processed = {
    "train_img": train_dir / "af_train_img_seqtoseq_alll_1i_1.npy",
    "train_lbl": train_dir / "af_train_label_seqtoseq_alll_1i_1.npy",
    "val_img":   val_dir   / "af_val_img_seqtoseq_alll_1i_1.npy",
    "val_lbl":   val_dir   / "af_val_label_seqtoseq_alll_1i_1.npy",
}


def processed_ready() -> bool:
    return all(p.exists() for p in required_processed.values())


def find_raw_root(base: Path):
    """Return base if it contains VIIRS_Day subdirectories, else None."""
    if not base.exists():
        return None
    return base if next(base.rglob("VIIRS_Day"), None) is not None else None


def patch_ts_satfire_repo(repo_dir: Path, raw_root: Path):
    """Patch TS-SatFire repo paths and fix common issues."""
    # 1) dataset_gen_afba.py — update raw data root
    gen_py = repo_dir / "dataset_gen_afba.py"
    code = gen_py.read_text(encoding="utf-8")
    raw_prefix = str(raw_root).replace("\\", "/") + "/"
    code = code.replace("/home/z/h/zhao2/CalFireMonitoring/data/", raw_prefix)
    gen_py.write_text(code, encoding="utf-8")

    # 2) satimg_dataset_processor.py — path compat + empty-list guard
    processor = repo_dir / "satimg_dataset_processor" / "satimg_dataset_processor.py"
    pcode = processor.read_text(encoding="utf-8")
    if "import os" not in pcode:
        pcode = "import os\n" + pcode

    replacements = [
        ("os.mkdir(save_path)", "os.makedirs(save_path, exist_ok=True)"),
        ("location + '/*_VIIRS_Day.tif'",   "location + '/VIIRS_Day/*_VIIRS_Day.tif'"),
        ("location + '/*_VIIRS_Night.tif'",  "location + '/VIIRS_Night/*_VIIRS_Night.tif'"),
        ('location + "/*_VIIRS_Day.tif"',    'location + "/VIIRS_Day/*_VIIRS_Day.tif"'),
        ('location + "/*_VIIRS_Night.tif"',  'location + "/VIIRS_Night/*_VIIRS_Night.tif"'),
        ("os.path.join(location, '*_VIIRS_Day.tif')",   "os.path.join(location, 'VIIRS_Day', '*_VIIRS_Day.tif')"),
        ('os.path.join(location, "*_VIIRS_Day.tif")',   'os.path.join(location, "VIIRS_Day", "*_VIIRS_Day.tif")'),
        ("os.path.join(location, '*_VIIRS_Night.tif')", "os.path.join(location, 'VIIRS_Night', '*_VIIRS_Night.tif')"),
        ('os.path.join(location, "*_VIIRS_Night.tif")', 'os.path.join(location, "VIIRS_Night", "*_VIIRS_Night.tif")'),
    ]
    for old, new in replacements:
        pcode = pcode.replace(old, new)

    # Guard against empty file lists
    lines = pcode.split("\n")
    for target_line in [
        "array_day, meta = self.read_tiff(file_list[0])",
        "array_night, meta = self.read_tiff(file_list[0])",
    ]:
        for i, line in enumerate(lines):
            if target_line in line:
                indent = len(line) - len(line.lstrip())
                if i > 0 and lines[i - 1].strip().startswith("if not file_list"):
                    break
                lines.insert(i,     " " * indent + "if not file_list:")
                lines.insert(i + 1, " " * (indent + 4) + 'print(f"Skipping {id} — empty file list")')
                lines.insert(i + 2, " " * (indent + 4) + "return")
                break

    processor.write_text("\n".join(lines), encoding="utf-8")


def preprocess_raw(raw_root: Path):
    """Clone TS-SatFire repo, patch it, run preprocessing → af_*.npy."""
    print("\n[TS] Preprocessing raw data → af_*.npy ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "rasterio"],
        check=False, capture_output=True,
    )

    repo_dir = Path("/content/TS-SatFire")
    if repo_dir.exists():
        shutil.rmtree(repo_dir)

    subprocess.run(
        ["git", "clone", "https://github.com/zhaoyutim/TS-SatFire.git", str(repo_dir)],
        check=True,
    )
    patch_ts_satfire_repo(repo_dir, raw_root)

    for mode in ["train", "val"]:
        print(f"[TS] Generating {mode} set ...")
        subprocess.run(
            [sys.executable, "dataset_gen_afba.py",
             "-mode", mode, "-ts", "1", "-it", "1", "-uc", "af"],
            cwd=str(repo_dir), check=True,
        )

    # Locate output npy files
    dataset_out = repo_dir / "dataset"

    def pick(exact: str, glob: str):
        hits = list(dataset_out.rglob(exact))
        if hits:
            return hits[0]
        hits = sorted(dataset_out.rglob(glob))
        return hits[0] if hits else None

    sources = {
        "train_img": pick("af_train_img_seqtoseq_alll_1i_1.npy",  "af_train_img_*.npy"),
        "train_lbl": pick("af_train_label_seqtoseq_alll_1i_1.npy", "af_train_label_*.npy"),
        "val_img":   pick("af_val_img_seqtoseq_alll_1i_1.npy",    "af_val_img_*.npy"),
        "val_lbl":   pick("af_val_label_seqtoseq_alll_1i_1.npy",  "af_val_label_*.npy"),
    }
    missing = [k for k, v in sources.items() if v is None]
    if missing:
        raise RuntimeError(f"Preprocessing failed — missing outputs: {missing}")

    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)
    for key, src in sources.items():
        dst = required_processed[key]
        shutil.copy2(src, dst)
        print(f"[TS] Saved: {dst}")


# ── Status ───────────────────────────────────────────────────────────────
satfire_ok = processed_ready()
hrrr_ok    = clim_dir.exists() and len(list(clim_dir.glob("doy_*.npy"))) >= 25
raw_ok     = find_raw_root(raw_dir) is not None

print(f"Processed data ready : {satfire_ok}  ({train_dir})")
print(f"Raw TS-SatFire ready : {raw_ok}  ({raw_dir})")
print(f"HRRR climatology     : {hrrr_ok}  ({clim_dir})")

# ═══════════════════════════════════════════════════════════════════════
# 1. TS-SatFire — preprocess from raw if needed
# ═══════════════════════════════════════════════════════════════════════
if satfire_ok and not FORCE_REPROCESS:
    print("\n[1/2] Processed data found — skipping preprocessing.")
else:
    raw_root = find_raw_root(raw_dir)
    if raw_root is None:
        raise RuntimeError(
            f"No raw TS-SatFire data found at {raw_dir}.\n"
            "Please ensure ts-satfire_raw/ is uploaded to Drive under "
            "MyDrive/FireST-GraphNet/ts-satfire_raw/"
        )
    print(f"\n[1/2] Raw data at: {raw_root}")
    preprocess_raw(raw_root)

    if not processed_ready():
        raise RuntimeError("Preprocessing completed but af_*.npy still missing.")
    print("[1/2] TS-SatFire processed data ready ✓")

# ═══════════════════════════════════════════════════════════════════════
# 2. HRRR DOY Climatology (29 representative dates, ~1 MB)
# ═══════════════════════════════════════════════════════════════════════
if not hrrr_ok:
    print("\n[2/2] Building HRRR DOY climatology ...")
    try:
        if importlib.util.find_spec("herbie") is None:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "herbie-data", "cfgrib"],
                check=True, capture_output=True,
            )
        from herbie import Herbie
        import numpy as np

        clim_dir.mkdir(parents=True, exist_ok=True)
        raw_hrrr = Path(DATA_ROOT) / "hrrr_cache" / "raw"
        raw_hrrr.mkdir(parents=True, exist_ok=True)

        VARIABLES = ["TMP:2 m", "UGRD:10 m", "VGRD:10 m", "RH:2 m"]
        REPR_DATES = {
            150: "2019-05-30", 156: "2019-06-05", 162: "2019-06-11",
            168: "2019-06-17", 174: "2019-06-23", 180: "2019-06-29",
            186: "2019-07-05", 192: "2019-07-11", 198: "2019-07-17",
            204: "2019-07-23", 210: "2019-07-29", 216: "2019-08-04",
            222: "2019-08-10", 228: "2019-08-16", 234: "2019-08-22",
            240: "2019-08-28", 246: "2019-09-03", 252: "2019-09-09",
            258: "2019-09-15", 264: "2019-09-21", 270: "2019-09-27",
            276: "2019-10-03", 282: "2019-10-09", 288: "2019-10-15",
            294: "2019-10-21", 300: "2019-10-27", 200: "2019-07-19",
            250: "2019-09-07", 306: "2019-11-02",
        }

        ok, fail = 0, 0
        for doy, date_str in sorted(REPR_DATES.items()):
            out_path = clim_dir / f"doy_{doy:03d}.npy"
            if out_path.exists():
                ok += 1
                continue
            try:
                H = Herbie(
                    f"{date_str} 18:00", model="hrrr", product="sfc", fxx=0,
                    save_dir=str(raw_hrrr), verbose=False,
                )
                bands = []
                for var in VARIABLES:
                    ds = H.xarray(var, remove_grib=False)
                    arr = ds[list(ds.data_vars)[0]].values.astype("float32")
                    cy, cx = arr.shape[0] // 2, arr.shape[1] // 2
                    patch = arr[cy - 128 : cy + 128, cx - 128 : cx + 128]
                    bands.append(patch)
                np.save(str(out_path), np.stack(bands))
                ok += 1
                print(f"  OK  DOY {doy:03d}")
            except Exception as ex:
                fail += 1
                print(f"  FAIL DOY {doy:03d}: {str(ex)[:80]}")

        print(f"HRRR: {ok} OK / {fail} failed")

        # Fill gaps with nearest-neighbor interpolation
        existing = sorted(clim_dir.glob("doy_*.npy"))
        if existing and fail > 0:
            for doy in range(150, 331):
                fp = clim_dir / f"doy_{doy:03d}.npy"
                if not fp.exists():
                    nearest = min(existing,
                                  key=lambda p: abs(int(p.stem.split("_")[1]) - doy))
                    shutil.copy(str(nearest), str(fp))
            print(f"Gap-filled to {len(list(clim_dir.glob('doy_*.npy')))} DOY files")

    except Exception as ex:
        print(f"HRRR download failed ({ex}) — training will use dataset bands as weather proxy.")
        clim_dir.mkdir(parents=True, exist_ok=True)
else:
    print("[2/2] HRRR climatology exists — skipping.")

# ═══════════════════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════════════════
# Check LULC static features
lulc_train = train_dir / "lulc_train.npy"
lulc_val   = val_dir   / "lulc_val.npy"
lulc_ok    = lulc_train.exists() or lulc_val.exists()
print(f"  ESRI LULC static   : {lulc_ok}  (placeholder zeros if missing — model still trains)")

# Check FirePred auxiliary data (19ch weather/topo/fuel)
firepred_train = sorted(train_dir.glob("firepred_*.npy"))
firepred_ok = len(firepred_train) > 0
print(f"  FirePred auxiliary  : {firepred_ok}  (placeholder zeros if missing — model still trains)")

print("\n[Note] WeatherNext2 requires restricted Google access — auto-fallback to zeros.")
print("\nData preparation complete ✓")
print(f"  dataset_train : {len(list(train_dir.glob('af_*_img_*.npy')))} img / "
      f"{len(list(train_dir.glob('af_*_label_*.npy')))} label")
print(f"  dataset_val   : {len(list(val_dir.glob('af_*_img_*.npy')))} img / "
      f"{len(list(val_dir.glob('af_*_label_*.npy')))} label")
print(f"  HRRR DOY      : {len(list(clim_dir.glob('doy_*.npy')))} files")
print(f"  FirePred aux  : {'found' if firepred_ok else 'placeholder (zeros)'}")
print(f"  LULC static   : {'found' if lulc_ok else 'placeholder (zeros)'}")

## 2. Load & Inspect Dataset

In [ ]:
import numpy as np
from pathlib import Path

train_dir = Path(DATA_ROOT) / "dataset_train"
val_dir   = Path(DATA_ROOT) / "dataset_val"

for name, d in [("Train", train_dir), ("Val", val_dir)]:
    if not d.exists():
        print(f"{name} dir not found: {d}")
        continue
    img_files = sorted(d.glob("af_*_img_*.npy"))
    lbl_files = sorted(d.glob("af_*_label_*.npy"))
    if img_files:
        img = np.load(str(img_files[0]), mmap_mode="r")
        lbl = np.load(str(lbl_files[0]), mmap_mode="r")
        print(f"{name:5s}  images: {img.shape}  labels: {lbl.shape}")
    else:
        print(f"{name:5s}  no af_*.npy found — run Step 1b first.")

## 3. Visualise Samples

In [ ]:
import matplotlib.pyplot as plt

img_files = sorted(train_dir.glob("af_*_img_*.npy"))
lbl_files = sorted(train_dir.glob("af_*_label_*.npy"))

if img_files:
    sample_idx = min(100, np.load(str(img_files[0]), mmap_mode="r").shape[0] - 1)
    img_data = np.load(str(img_files[0]), mmap_mode="r")
    lbl_data = np.load(str(lbl_files[0]), mmap_mode="r")

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(img_data[sample_idx, 0, 0], cmap="viridis")
    axes[0].set_title(f"Image band 0  (sample {sample_idx})")

    axes[1].imshow(lbl_data[sample_idx, 1, 0], cmap="hot", vmin=0, vmax=1)
    axes[1].set_title("Fire mask (label ch 1)")

    fire_frac = lbl_data[sample_idx, 1, 0].mean()
    axes[2].text(0.1, 0.5,
                 f"Fire fraction: {fire_frac:.4f}\n"
                 f"Image shape: {img_data.shape}\n"
                 f"Label shape: {lbl_data.shape}",
                 fontsize=12, transform=axes[2].transAxes)
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No data to visualise — run Step 1b first.")

## 4. Configuration

In [ ]:
import yaml

# Load base config
cfg_path = Path(CODE_ROOT) / "firest_graphnet" / "configs" / "base.yaml"
with open(cfg_path, encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# Override for Colab / local
cfg["data"]["root"] = str(DATA_ROOT)
cfg["data"]["num_workers"] = 2 if IN_COLAB else 0
cfg["device"] = "auto"

# ── Choose mode ──────────────────────────────────────────────────────────
MODE    = "forecast"   # "legacy" or "forecast"
HORIZON = 20           # 1 for single-step, 20 for multi-horizon
EPOCHS  = 30

cfg["training"]["epochs"]     = EPOCHS
cfg["training"]["batch_size"] = 2   # small for Colab GPU memory

if MODE == "forecast":
    cfg.setdefault("forecast", {})
    cfg["forecast"]["T_in"]     = 10
    cfg["forecast"]["horizon"]  = HORIZON

    # ── Multi-source encoder input (per TS-SatFire paper) ────────────────
    #   fire_channels  = 8   VIIRS: Day 6 spectral (I1,I2,I3,M13,M15,M11)
    #                        + Night 2 thermal (I4, I5)
    #   aux_channels   = 19  FirePred: DEM, slope, aspect, canopy, ERC,
    #                        temp, dewpoint, RH, precip, wind speed/dir,
    #                        fuel moisture, PDSI, fuel model, dead fuel,
    #                        VPD, NDVI, soil temp, snow
    #   static_channels = 1  ESRI LULC land cover
    #   → total encoder in_channels = 8 + 19 + 1 = 28

    FIRE_CHANNELS    = 8   # VIIRS Day(6) + Night(2) satellite imagery
    AUX_CHANNELS     = 19  # FirePred weather / topography / fuel
    STATIC_CHANNELS  = 1   # ESRI LULC vegetation / land cover
    WEATHER_CHANNELS = 4   # HRRR future weather for FiLM decoder

    cfg["forecast"]["fire_channels"]    = FIRE_CHANNELS
    cfg["forecast"]["aux_channels"]     = AUX_CHANNELS
    cfg["forecast"]["static_channels"]  = STATIC_CHANNELS
    cfg["forecast"]["weather_channels"] = WEATHER_CHANNELS
    cfg["forecast"]["in_channels"]      = FIRE_CHANNELS + AUX_CHANNELS + STATIC_CHANNELS
    cfg["forecast"]["weather_provider"] = "hrrr"
    cfg["forecast"]["weather_emb_dim"]  = 64

    print(f"Encoder input: fire({FIRE_CHANNELS}) + aux({AUX_CHANNELS}) + static({STATIC_CHANNELS}) = {cfg['forecast']['in_channels']} channels")
    print(f"Decoder FiLM: HRRR future weather ({WEATHER_CHANNELS} channels)")

print(yaml.dump(cfg, default_flow_style=False))

## 5. Build Dataset & Model

In [ ]:
import torch
from torch.utils.data import DataLoader
from firest_graphnet.utils.seed import seed_everything

seed_everything(cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")

if MODE == "forecast":
    from firest_graphnet.data.dataset_forecast import build_forecast_datasets
    from firest_graphnet.data.weather_provider import build_weather_provider
    from firest_graphnet.data.firepred_provider import build_firepred_provider
    from firest_graphnet.data.static_provider import build_static_provider

    fcfg = cfg["forecast"]

    # ── Weather provider (HRRR → placeholder fallback) ────────────────
    #    Used ONLY for FiLM decoder future weather conditioning
    wp = build_weather_provider(mode=fcfg["weather_provider"], data_root=DATA_ROOT)
    actual_weather_channels = wp.Cw
    print(f"Weather provider   : {type(wp).__name__}  ({actual_weather_channels} ch — FiLM decoder)")

    # ── FirePred auxiliary provider (19ch → placeholder fallback) ──────
    #    Historical weather/topo/fuel from FirePred per-date TIFFs
    fp = build_firepred_provider(data_root=DATA_ROOT, channels=fcfg.get("aux_channels", 19))
    actual_aux_channels = fp.channels
    print(f"FirePred provider  : {type(fp).__name__}  ({actual_aux_channels} ch — encoder aux)")

    # ── Static feature provider (LULC → placeholder fallback) ─────────
    sp = build_static_provider(data_root=DATA_ROOT, channels=fcfg["static_channels"])
    actual_static_channels = sp.channels
    print(f"Static provider    : {type(sp).__name__}  ({actual_static_channels} ch — encoder static)")

    # ── Recompute total encoder channels after potential fallbacks ─────
    actual_in_channels = fcfg["fire_channels"] + actual_aux_channels + actual_static_channels
    print(f"\nTotal encoder in   : {fcfg['fire_channels']}(fire) + {actual_aux_channels}(aux) + {actual_static_channels}(static) = {actual_in_channels}")

    # ── Build datasets ────────────────────────────────────────────────
    train_ds, val_ds = build_forecast_datasets(
        data_root=DATA_ROOT,
        T_in=fcfg["T_in"],
        horizon=fcfg["horizon"],
        label_channel=cfg["data"].get("label_channel", 1),
        seed=cfg["seed"],
        weather_provider=wp,
        firepred_provider=fp,
        static_provider=sp,
        img_as_input=True,
    )
else:
    from firest_graphnet.data.dataset import build_datasets
    actual_weather_channels = 0
    actual_aux_channels = 0
    actual_static_channels = 0
    actual_in_channels = cfg["model"].get("in_channels", 8)
    train_ds, val_ds = build_datasets(
        data_root=DATA_ROOT,
        T_in=cfg["data"]["T_in"],
        T_out=cfg["data"].get("T_out", 1),
        label_channel=cfg["data"].get("label_channel", 1),
        seed=cfg["seed"],
    )

print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}")

bs = cfg["training"]["batch_size"]
train_loader = DataLoader(
    train_ds, batch_size=bs, shuffle=True,
    num_workers=cfg["data"]["num_workers"], drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=bs, shuffle=False,
    num_workers=cfg["data"]["num_workers"],
)

In [ ]:
from firest_graphnet.models.firest_graphnet import FireSTGraphNet

mcfg = cfg["model"]
dcfg = cfg["data"]

model_kwargs = dict(
    encoder_channels=mcfg.get("encoder_channels", [32, 64, 128]),
    use_cbam=mcfg.get("use_cbam", True),
    patch_size=mcfg.get("patch_size", 8),
    k_neighbors=mcfg.get("k_neighbors", 8),
    gnn_hidden=mcfg.get("gnn_hidden", 128),
    gnn_layers=mcfg.get("gnn_layers", 2),
    lstm_hidden=mcfg.get("lstm_hidden", 256),
    lstm_layers=mcfg.get("lstm_layers", 2),
    lstm_dropout=mcfg.get("lstm_dropout", 0.1),
    decoder_channels=mcfg.get("decoder_channels", [128, 64, 32]),
    img_h=dcfg.get("img_size", 256),
    img_w=dcfg.get("img_size", 256),
)

if MODE == "forecast":
    fcfg = cfg["forecast"]
    model = FireSTGraphNet(
        # Encoder: fire(8) + aux(19) + static(1) = 28 total channels
        in_channels=actual_in_channels,
        out_channels=1,
        forecast_mode=True,
        # FiLM decoder: HRRR future weather conditioning (4ch)
        weather_channels=actual_weather_channels,
        weather_emb_dim=fcfg.get("weather_emb_dim", 64),
        **model_kwargs,
    )
    print(f"Encoder in_channels      : {actual_in_channels}")
    print(f"  → fire (VIIRS)         : {fcfg['fire_channels']}  (Day I1-M11 + Night I4,I5)")
    print(f"  → aux  (FirePred)      : {actual_aux_channels}  (weather/topo/fuel)")
    print(f"  → static (ESRI LULC)   : {actual_static_channels}")
    print(f"Decoder weather (HRRR)   : {actual_weather_channels}  ({type(wp).__name__})")
else:
    model = FireSTGraphNet(
        in_channels=mcfg.get("in_channels", 8),
        out_channels=mcfg.get("out_channels", 1),
        **model_kwargs,
    )

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params:,} trainable parameters")
print(f"Forecast mode: {model.forecast_mode}")

## 6. Train

In [ ]:
from firest_graphnet.utils.io import ensure_dir, save_yaml

out_dir = ensure_dir(Path(CODE_ROOT) / "outputs")
save_yaml(cfg, out_dir / "config.yaml")
print(f"Checkpoint dir: {out_dir}")

if MODE == "forecast":
    from firest_graphnet.training.trainer import ForecastTrainer
    trainer = ForecastTrainer(
        model=model, train_loader=train_loader, val_loader=val_loader,
        cfg=cfg, device=device, output_dir=out_dir,
    )
else:
    from firest_graphnet.training.trainer import Trainer
    trainer = Trainer(
        model=model, train_loader=train_loader, val_loader=val_loader,
        cfg=cfg, device=device, output_dir=out_dir,
    )

trainer.fit()

## 6b. Load Saved Checkpoint (skip training)

If you have a previously trained model, run this cell to load it directly.

In [ ]:
import torch
from pathlib import Path

ckpt_path = Path(CODE_ROOT) / "outputs" / "best_model.pt"

if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()
    print(f"Loaded checkpoint: {ckpt_path}")
    print(f"  Epoch: {ckpt['epoch']}  |  Best Val IoU: {ckpt['best_val_iou']:.4f}")
else:
    print(f"No checkpoint at: {ckpt_path}")
    print("Run Step 6 to train first.")

## 7. Evaluate

In [ ]:
from firest_graphnet.training.metrics import (
    compute_metrics, compute_metrics_per_horizon, metrics_summary_str,
)

# Load best model
ckpt = torch.load(str(out_dir / "best_model.pt"), map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()
print(f"Best model from epoch {ckpt['epoch']}, val IoU = {ckpt['best_val_iou']:.4f}")

# Collect predictions
all_logits, all_targets = [], []
with torch.no_grad():
    for batch in val_loader:
        if MODE == "forecast":
            af_hist  = batch["af_hist"].to(device)
            aux_hist = batch["aux_hist"].to(device)
            w_fut    = batch["w_fut"].to(device)
            static   = batch["static"].to(device)
            logits   = model(af_hist, aux_hist=aux_hist, w_fut=w_fut, static=static)
            targets  = batch["af_fut"].to(device)
        else:
            logits  = model(batch["x_seq"].to(device))
            targets = batch["y"].to(device)
        all_logits.append(logits.cpu())
        all_targets.append(targets.cpu())

all_logits  = torch.cat(all_logits)
all_targets = torch.cat(all_targets)

# Aggregate metrics
agg = compute_metrics(all_logits, all_targets)
print("\nAggregate Metrics:")
for k, v in agg.items():
    print(f"  {k:>12s}: {v:.4f}")

# Per-horizon (forecast mode)
if MODE == "forecast" and all_logits.dim() == 5:
    per_h = compute_metrics_per_horizon(all_logits, all_targets)
    print("\nPer-Horizon Metrics:")
    print(metrics_summary_str(per_h))

## 8. Visualise Predictions

In [ ]:
import matplotlib.pyplot as plt


def plot_forecast(logits, targets, sample_idx=0, horizons=[0, 4, 9, 14, 19]):
    """Predicted vs ground-truth AF at selected horizons."""
    probs = torch.sigmoid(logits)
    horizons = [h for h in horizons if h < probs.shape[1]]

    fig, axes = plt.subplots(2, len(horizons), figsize=(4 * len(horizons), 7))
    if len(horizons) == 1:
        axes = axes.reshape(2, 1)

    for i, h in enumerate(horizons):
        axes[0, i].imshow(probs[sample_idx, h, 0].numpy(), cmap="hot", vmin=0, vmax=1)
        axes[0, i].set_title(f"Pred h={h}")
        axes[0, i].axis("off")

        axes[1, i].imshow(targets[sample_idx, h, 0].numpy(), cmap="hot", vmin=0, vmax=1)
        axes[1, i].set_title(f"GT h={h}")
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Prediction", fontsize=12)
    axes[1, 0].set_ylabel("Ground Truth", fontsize=12)
    plt.suptitle("Multi-Horizon Fire Prediction", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_legacy(logits, targets, sample_idx=0):
    probs = torch.sigmoid(logits)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(probs[sample_idx, 0].numpy(), cmap="hot", vmin=0, vmax=1)
    axes[0].set_title("Prediction")
    axes[1].imshow(targets[sample_idx, 0].numpy(), cmap="hot", vmin=0, vmax=1)
    axes[1].set_title("Ground Truth")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


if MODE == "forecast":
    plot_forecast(all_logits, all_targets, sample_idx=0)
else:
    plot_legacy(all_logits, all_targets, sample_idx=0)

## 9. Physics Constraint Analysis

In [ ]:
if MODE == "forecast":
    from firest_graphnet.physics.ba_update import BAStateUpdate
    from firest_graphnet.physics.schedules import PINNSchedule

    # BA trajectory for a sample
    ba_updater = BAStateUpdate(gamma=cfg.get("pinn", {}).get("gamma", 0.3))
    sample = val_ds[0]
    af_logits = all_logits[0:1]                        # (1, H, 1, Hp, Wp)
    ba_init = sample["ba_init"].unsqueeze(0)            # (1, 1, Hp, Wp)
    ba_seq = ba_updater(af_logits, ba_init)             # (1, H, 1, Hp, Wp)

    # Plot BA evolution
    H = ba_seq.shape[1]
    steps = [0, H // 4, H // 2, 3 * H // 4, H - 1]
    steps = sorted(set(s for s in steps if s < H))

    fig, axes = plt.subplots(1, len(steps), figsize=(4 * len(steps), 3.5))
    if len(steps) == 1:
        axes = [axes]
    for i, s in enumerate(steps):
        axes[i].imshow(ba_seq[0, s, 0].detach().numpy(), cmap="Reds", vmin=0, vmax=1)
        axes[i].set_title(f"BA h={s}")
        axes[i].axis("off")
    plt.suptitle("Burned Area Evolution (monotonic)", fontsize=13)
    plt.tight_layout()
    plt.show()

    # PINN weight schedule
    pcfg = cfg.get("pinn", {})
    schedule = PINNSchedule(
        **{k: v for k, v in pcfg.items()
           if k in ["beta_max", "eta_max", "xi_max", "warmup_epochs", "start_epoch"]}
    )
    epochs = list(range(1, cfg["training"]["epochs"] + 1))
    betas = [schedule.get_weights(e)["beta"] for e in epochs]
    etas  = [schedule.get_weights(e)["eta"]  for e in epochs]
    xis   = [schedule.get_weights(e)["xi"]   for e in epochs]

    plt.figure(figsize=(8, 3))
    plt.plot(epochs, betas, label="β (PDE)")
    plt.plot(epochs, etas,  label="η (mono)")
    plt.plot(epochs, xis,   label="ξ (mask)")
    plt.xlabel("Epoch")
    plt.ylabel("Weight")
    plt.title("PINN Loss Weight Schedule")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Physics analysis is only available in forecast mode.")

## 10. Export & Save

In [ ]:
import csv

# Save per-horizon CSV (forecast mode)
if MODE == "forecast" and all_logits.dim() == 5:
    csv_path = out_dir / "metrics_by_horizon.csv"
    per_h = compute_metrics_per_horizon(all_logits, all_targets)
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["horizon", "iou", "f1", "precision", "recall"])
        writer.writeheader()
        for h in sorted(per_h.keys()):
            writer.writerow({"horizon": h, **per_h[h]})
    print(f"Saved: {csv_path}")

# Download outputs if on Colab
if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("outputs_colab", "zip", str(out_dir))
    files.download("outputs_colab.zip")
    print("Download started!")
else:
    print(f"Outputs saved to: {out_dir}")